# 03 Prepare Global Causal Analysis Dataset

This notebook prepares the NSW benchmark data for global causal analysis (Task 3).
Upload all required files to Colab first, then run all cells top to bottom.

**Required files (upload to Colab before running):**
- `adj_matrix_67.npy`
- `adj_matrix_67_binary.npy`
- `node_order_67.npy`
- `centrality_rankings.csv`
- `final_incidents_with_weather.csv`
- `final_hourly_flow_allfeature_with_timefeat.csv`

## 1. Import libraries

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Output folder for saved files
os.makedirs('data_global_causal', exist_ok=True)
print('Libraries imported.')
print('Output folder: data_global_causal/')

Libraries imported.
Output folder: data_global_causal/


## 3. Load all files

In [23]:
# Graph files
adj_67        = np.load('adj_matrix_67.npy')
adj_67_binary = np.load('adj_matrix_67_binary.npy')
node_order_67 = np.load('node_order_67.npy', allow_pickle=True).astype(str)

# Table files
centrality        = pd.read_csv('centrality_rankings.csv')
incidents_weather = pd.read_csv('final_incidents_with_weather.csv', low_memory=False)
flow_all          = pd.read_csv('final_hourly_flow_allfeature_with_timefeat.csv', low_memory=False)

print('Graph files loaded')
print('  adj_67:         ', adj_67.shape)
print('  adj_67_binary:  ', adj_67_binary.shape)
print('  node_order_67:  ', node_order_67.shape)
print()
print('Table files loaded')
print('  centrality:         ', centrality.shape)
print('  incidents_weather:  ', incidents_weather.shape)
print('  flow_all:           ', flow_all.shape)

Graph files loaded
  adj_67:          (67, 67)
  adj_67_binary:   (67, 67)
  node_order_67:   (67,)

Table files loaded
  centrality:          (67, 14)
  incidents_weather:   (3197, 26)
  flow_all:            (1007400, 31)


## 4. Standardise identifiers and timestamps

In [24]:
# Standardise station IDs to string
flow_all['station_id']          = flow_all['station_id'].astype(str).str.strip()
incidents_weather['station_id'] = incidents_weather['station_id'].astype(str).str.strip()
centrality['station_id']        = centrality['station_id'].astype(str).str.strip()
node_order_67                   = node_order_67.astype(str)

# Parse timestamps
flow_all['timestamp']           = pd.to_datetime(flow_all['timestamp'],   errors='coerce')
incidents_weather['match_hour'] = pd.to_datetime(incidents_weather['match_hour'], errors='coerce')

print('Station IDs standardised')
print('Timestamps parsed')
print('Flow date range:     ', flow_all['timestamp'].min(), 'to', flow_all['timestamp'].max())
print('Incident date range: ', incidents_weather['match_hour'].min(), 'to', incidents_weather['match_hour'].max())

Station IDs standardised
Timestamps parsed
Flow date range:      2025-01-01 00:00:00 to 2025-12-31 23:00:00
Incident date range:  2025-01-01 04:00:00 to 2025-12-31 06:00:00


## 5. Inspect datasets

In [25]:
print('FLOW COLUMNS')
print(flow_all.columns.tolist())
print()
print('INCIDENT-WEATHER COLUMNS')
print(incidents_weather.columns.tolist())
print()
print('Flow stations:    ', flow_all['station_id'].nunique())
print('Incident stations:', incidents_weather['station_id'].nunique())
print('Hazard types:     ', incidents_weather['hazard_type'].unique())

FLOW COLUMNS
['station_id', 'timestamp', 'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'is_holiday', 'hour', 'total_flow', 'wgs84_latitude', 'wgs84_longitude', 'road_name', 'suburb', 'post_code', 'device_type', 'quality_rating', 'lane_count', 'road_functional_hierarchy', 'distance_to_intersection', 'incident_id', 'incident_type', 'is_major_incident', 'impact_sequence_hour', 'precipitation', 'weather_code', 'apparent_temperature', 'temperature_2m', 'wind_gusts_10m', 'relative_humidity', 'incident_count', 'is_anomaly']

INCIDENT-WEATHER COLUMNS
['station_id', 'Abs PM_x', 'Abs PM_y', 'incident_id', 'dis', 'hazard_type', 'incident_kind', 'match_hour', 'calculated_duration_hours', 'impact_sequence_hour', 'is_major_incident', 'is_local_road', 'speed_limit', 'advice_a', 'advice_b', 'queue_length_km', 'traffic_volume_desc', 'affected_direction', 'temperature_2m', 'rain', 'precipitation', 'weather_code', 'apparent_temperature', 'relative_humidity', 'wind_gusts_10m', 'dew_point_2m']

Flow

## 6. Check station consistency

In [26]:
flow_ids     = set(flow_all['station_id'])
incident_ids = set(incidents_weather['station_id'])
node67_ids   = set(node_order_67)

print('Flow stations:      ', len(flow_ids))
print('Incident stations:  ', len(incident_ids))
print('Node order 67:      ', len(node67_ids))
print()
print('Flow == node_order_67:', flow_ids == node67_ids)

missing = node67_ids - flow_ids
print('Stations in graph but not in flow:', missing if missing else 'None')

Flow stations:       115
Incident stations:   67
Node order 67:       67

Flow == node_order_67: False
Stations in graph but not in flow: None


## 7. Check flow data quality

In [27]:
quality = (
    flow_all
    .groupby('station_id')['total_flow']
    .agg(
        n_records  = 'count',
        mean_flow  = 'mean',
        zero_rate  = lambda x: (x == 0).mean(),
        nan_rate   = lambda x: x.isna().mean()
    )
    .reset_index()
    .sort_values('zero_rate', ascending=False)
)

print('Flow data quality per station (top 10):')
print(quality.head(10).to_string(index=False))

Flow data quality per station (top 10):
station_id  n_records  mean_flow  zero_rate  nan_rate
     T0296       8760   0.057192   0.993836       0.0
      6114       8760   1.969064   0.960616       0.0
      6109       8760   7.323174   0.954452       0.0
     32029       8760  22.499658   0.945662       0.0
      6110       8760   7.902511   0.936416       0.0
      6113       8760   5.786644   0.900913       0.0
     10011       8760  82.531735   0.867466       0.0
     22001       8760  56.476826   0.864155       0.0
     T0289       8760  14.600799   0.858219       0.0
     T0288       8760  20.624087   0.829680       0.0


## 8. Inspect incident dataset

In [28]:
print('INCIDENT DATASET')
print('Rows:             ', len(incidents_weather))
print('Unique incidents: ', incidents_weather['incident_id'].nunique())
print('Unique stations:  ', incidents_weather['station_id'].nunique())
print()
print('Hazard types:')
print(incidents_weather['hazard_type'].value_counts())
print()
print('Major incidents:')
print(incidents_weather['is_major_incident'].value_counts())
print()
print('Weather summary:')
print(incidents_weather[['precipitation', 'apparent_temperature',
                          'wind_gusts_10m', 'relative_humidity']].describe())

INCIDENT DATASET
Rows:              3197
Unique incidents:  1628
Unique stations:   67

Hazard types:
hazard_type
BREAKDOWN                         1136
CRASH                              688
SPECIAL EVENT                      576
SCHEDULED ROADWORK                 145
HAZARD                             144
TRAFFIC LIGHTS BLACKED OUT         138
EMERGENCY ROADWORK                 103
CHANGED TRAFFIC CONDITIONS          86
ADVERSE WEATHER                     73
TRAFFIC LIGHTS FLASHING YELLOW      69
BUILDING FIRE                       16
FLOODING                            10
BURST WATER MAIN                     4
BUSHFIRE                             4
HEAVY TRAFFIC                        2
LATE FINISHING ROADWORK              1
HOLIDAY TRAFFIC                      1
GRASS FIRE                           1
Name: count, dtype: int64

Major incidents:
is_major_incident
0    2344
1     853
Name: count, dtype: int64

Weather summary:
       precipitation  apparent_temperature  wind_gusts_10m

## 9. Aggregate daily flow variables

Following TraffiDent MM-DAG approach: aggregate hourly data to daily
samples at each station. One daily observation per station = one sample.

In [29]:
flow_all['date']       = flow_all['timestamp'].dt.date
flow_all['is_weekend'] = flow_all['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

flow_daily = (
    flow_all
    .groupby(['station_id', 'date'])
    .agg(
        total_flow_daily    = ('total_flow', 'sum'),
        mean_flow_hourly    = ('total_flow', 'mean'),
        is_major_incident   = ('is_major_incident', 'max'),
        impact_hours        = ('impact_sequence_hour', lambda x: (x >= 0).sum()),
        precipitation_daily = ('precipitation', 'sum'),
        is_weekend          = ('is_weekend', 'max'),
    )
    .reset_index()
)

print('Daily flow matrix shape:', flow_daily.shape)
print(flow_daily.head(3))

Daily flow matrix shape: (41975, 8)
  station_id        date  total_flow_daily  mean_flow_hourly  \
0     100001  2025-01-01             351.0         14.625000   
1     100001  2025-01-02            1493.0         62.208333   
2     100001  2025-01-03            7200.0        300.000000   

   is_major_incident  impact_hours  precipitation_daily  is_weekend  
0                  0             0                  0.2           0  
1                  0             0                  0.1           0  
2                  0             0                  0.4           0  


## 10. Aggregate daily incident counts per hazard type

In [30]:
incidents_weather['date'] = incidents_weather['match_hour'].dt.date

incident_daily = (
    incidents_weather
    .groupby(['station_id', 'date', 'hazard_type'])['incident_id']
    .nunique()
    .reset_index()
    .rename(columns={'incident_id': 'incident_count'})
)

incident_pivot = incident_daily.pivot_table(
    index=['station_id', 'date'],
    columns='hazard_type',
    values='incident_count',
    aggfunc='sum',
    fill_value=0
).reset_index()

incident_pivot.columns.name = None
incident_pivot.columns = [
    c.lower().replace(' ', '_') for c in incident_pivot.columns
]

print('Incident pivot shape:', incident_pivot.shape)
print('Columns:', incident_pivot.columns.tolist())

Incident pivot shape: (1683, 20)
Columns: ['station_id', 'date', 'adverse_weather', 'breakdown', 'building_fire', 'burst_water_main', 'bushfire', 'changed_traffic_conditions', 'crash', 'emergency_roadwork', 'flooding', 'grass_fire', 'hazard', 'heavy_traffic', 'holiday_traffic', 'late_finishing_roadwork', 'scheduled_roadwork', 'special_event', 'traffic_lights_blacked_out', 'traffic_lights_flashing_yellow']


## 11. Merge all variables into global matrix

In [31]:
hazard_cols = [
    c for c in incident_pivot.columns
    if c not in ['station_id', 'date']
]

global_df = flow_daily.merge(
    incident_pivot, on=['station_id', 'date'], how='left'
)
global_df[hazard_cols] = global_df[hazard_cols].fillna(0)

global_df = global_df.merge(
    centrality[['station_id', 'composite_score',
                'degree_cent', 'between_cent', 'close_cent']],
    on='station_id', how='left'
)

print('Global variable matrix shape:', global_df.shape)
print('Columns:', global_df.columns.tolist())
print()
print('Missing values:')
print(global_df.isnull().sum()[global_df.isnull().sum() > 0])

Global variable matrix shape: (41975, 30)
Columns: ['station_id', 'date', 'total_flow_daily', 'mean_flow_hourly', 'is_major_incident', 'impact_hours', 'precipitation_daily', 'is_weekend', 'adverse_weather', 'breakdown', 'building_fire', 'burst_water_main', 'bushfire', 'changed_traffic_conditions', 'crash', 'emergency_roadwork', 'flooding', 'grass_fire', 'hazard', 'heavy_traffic', 'holiday_traffic', 'late_finishing_roadwork', 'scheduled_roadwork', 'special_event', 'traffic_lights_blacked_out', 'traffic_lights_flashing_yellow', 'composite_score', 'degree_cent', 'between_cent', 'close_cent']

Missing values:
composite_score    17520
degree_cent        17520
between_cent       17520
close_cent         17520
dtype: int64


## 12. Build numpy array for MM-DAG / DYNOTEARS

Select final variables following TraffiDent Table 7:
- Meta-features: time, weather, road centrality
- Incident variables: hazard type counts
- Traffic statistics: flow, incident impact

In [32]:
meta_cols     = ['is_weekend', 'precipitation_daily',
                 'degree_cent', 'between_cent', 'close_cent']
incident_cols = hazard_cols
traffic_cols  = ['mean_flow_hourly', 'is_major_incident', 'impact_hours']

feature_cols  = meta_cols + incident_cols + traffic_cols
categories    = (
    ['meta']     * len(meta_cols) +
    ['incident'] * len(incident_cols) +
    ['traffic']  * len(traffic_cols)
)

# Diagnose WHERE the NaNs come from before dropping, so we know what we lose
print('NaN count per feature column (before dropna):')
nan_counts = global_df[feature_cols].isnull().sum()
print(nan_counts[nan_counts > 0] if (nan_counts > 0).any() else '  none')
print()

# Keep station_id and date attached to the SAME rows we keep for X, so that
# downstream leave-one-station-out can map each X row back to its station.
keep_cols     = ['station_id', 'date'] + feature_cols
global_clean  = global_df[keep_cols].dropna(subset=feature_cols).reset_index(drop=True)
X             = global_clean[feature_cols].values.astype(np.float32)

print('Variable categories:')
print('  Meta-features  :', meta_cols)
print('  Incident vars  :', incident_cols)
print('  Traffic stats  :', traffic_cols)
print(f'\nFinal matrix shape: {X.shape}  (samples x variables)')
print(f'Dropped rows (NaN): {len(global_df) - len(global_clean)}'
      f'  ({100*(len(global_df)-len(global_clean))/len(global_df):.1f}% of rows)')


NaN count per feature column (before dropna):
degree_cent     17520
between_cent    17520
close_cent      17520
dtype: int64

Variable categories:
  Meta-features  : ['is_weekend', 'precipitation_daily', 'degree_cent', 'between_cent', 'close_cent']
  Incident vars  : ['adverse_weather', 'breakdown', 'building_fire', 'burst_water_main', 'bushfire', 'changed_traffic_conditions', 'crash', 'emergency_roadwork', 'flooding', 'grass_fire', 'hazard', 'heavy_traffic', 'holiday_traffic', 'late_finishing_roadwork', 'scheduled_roadwork', 'special_event', 'traffic_lights_blacked_out', 'traffic_lights_flashing_yellow']
  Traffic stats  : ['mean_flow_hourly', 'is_major_incident', 'impact_hours']

Final matrix shape: (24455, 26)  (samples x variables)
Dropped rows (NaN): 17520  (41.7% of rows)


## 13. Save output files

In [33]:
import json

# Global variable table — save the SAME rows that make up X (i.e. after the
# NaN drop), WITH station_id and date attached. This guarantees the csv is
# row-aligned with global_X.npy, so leave-one-station-out in notebook 04 can
# map every X row back to its station. (Previously the full pre-dropna
# global_df was saved, which had more rows than X and broke the alignment.)
global_clean.to_csv('data_global_causal/global_variable_matrix.csv', index=False)

# Numpy array for DYNOTEARS
np.save('data_global_causal/global_X.npy', X)

# Variable names and categories
var_df = pd.DataFrame({'variable': feature_cols, 'category': categories})
var_df.to_csv('data_global_causal/variable_names.csv', index=False)

# Case summary
summary = {
    'n_stations'      : int(global_df['station_id'].nunique()),
    'n_days'          : int(global_df['date'].nunique()),
    'n_samples'       : int(len(global_clean)),
    'n_variables'     : int(len(feature_cols)),
    'n_meta_vars'     : int(len(meta_cols)),
    'n_incident_vars' : int(len(incident_cols)),
    'n_traffic_vars'  : int(len(traffic_cols)),
}
with open('data_global_causal/global_case_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Files saved to data_global_causal/')
print('  global_variable_matrix.csv')
print('  global_X.npy')
print('  variable_names.csv')
print('  global_case_summary.json')

Files saved to data_global_causal/
  global_variable_matrix.csv
  global_X.npy
  variable_names.csv
  global_case_summary.json


## 14. Summary

In [34]:
print('GLOBAL CAUSAL ANALYSIS — DATASET SUMMARY')
print('=' * 50)
print(f'Stations            : {global_df["station_id"].nunique()}')
print(f'Days covered        : {global_df["date"].nunique()}')
print(f'Total samples       : {len(global_clean)}')
print(f'Total variables     : {len(feature_cols)}')
print(f'  Meta-features     : {len(meta_cols)}')
print(f'  Incident vars     : {len(incident_cols)}')
print(f'  Traffic stats     : {len(traffic_cols)}')
print()
print('Variable list:')
for i, (v, c) in enumerate(zip(feature_cols, categories)):
    print(f'  [{i:02d}] {v:<35} ({c})')

GLOBAL CAUSAL ANALYSIS — DATASET SUMMARY
Stations            : 115
Days covered        : 365
Total samples       : 24455
Total variables     : 26
  Meta-features     : 5
  Incident vars     : 18
  Traffic stats     : 3

Variable list:
  [00] is_weekend                          (meta)
  [01] precipitation_daily                 (meta)
  [02] degree_cent                         (meta)
  [03] between_cent                        (meta)
  [04] close_cent                          (meta)
  [05] adverse_weather                     (incident)
  [06] breakdown                           (incident)
  [07] building_fire                       (incident)
  [08] burst_water_main                    (incident)
  [09] bushfire                            (incident)
  [10] changed_traffic_conditions          (incident)
  [11] crash                               (incident)
  [12] emergency_roadwork                  (incident)
  [13] flooding                            (incident)
  [14] grass_fire            

In [35]:
from google.colab import files
files.download('data_global_causal/global_X.npy')
files.download('data_global_causal/variable_names.csv')
files.download('data_global_causal/global_variable_matrix.csv')
files.download('data_global_causal/global_case_summary.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>